## Imports

In [ ]:
!pip install nltk transformers torch accelerate


Usage:   
  pip install [options] <requirement specifier> [package-index-options] ...
  pip install [options] -r <requirements file> [package-index-options] ...
  pip install [options] [-e] <vcs project url> ...
  pip install [options] [-e] <local project path> ...
  pip install [options] <archive url/path> ...

no such option: -!


   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   --------------------------- ------------ 1.0/1.6 MB 3.9 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 3.8 MB/s  0:00:00



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json, re, math

from datetime import datetime

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

nltk.download('punkt',     quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

## Model Loading

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer  = AutoTokenizer.from_pretrained(model_name)

model      = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
    
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=-1)
print("Model ready!")

p:\VS_code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 19088.64it/s]


Model ready!


## Tools

## Calculator Tool

In [ ]:
def calculator_tool(params: dict) -> dict:
    """
    Evaluates a math expression string.
    Params: { "expression": "10 * (3 + 2)" }
    Supports: +, -, *, /, **, sqrt, abs, round
    """
    try:
        expression = params.get("expression", "")
        
        if not expression:
            return {"error": "No expression provided."}
        
        safe_env = {
            "__builtins__": {},
            "sqrt": math.sqrt,
            "abs" : abs,
            "round": round,
            "pi"  : math.pi,
        }
        result = eval(expression, safe_env)
        return {"expression": expression, "result": round(result, 6)}
    
    except ZeroDivisionError:
        return {"error": "Division by zero is not allowed."}
    
    except Exception as e:
        return {"error": f"Invalid expression: {e}"}

## Text Processor Tool

In [ ]:
def text_processor_tool(params: dict) -> dict:
    """
    Analyzes text.
    Params: { "text": "...", "actions": ["word_count", "reading_time", "keywords", "summarize"] }
    """
    text    = params.get("text", "").strip()
    actions = params.get("actions", ["word_count"])
    
    if not text:
        return {"error": "No text provided."}
    
    stop_words = set(stopwords.words("english"))
    tokens     = word_tokenize(text)
    results    = {}

    if "word_count" in actions:
        results["word_count"] = len(tokens)

    if "reading_time" in actions:
        mins = len(tokens) / 200          # avg 200 wpm
        secs = int(mins * 60)
        results["reading_time"] = f"{secs} seconds (~{secs // 60}m {secs % 60}s)"

    if "keywords" in actions:
        clean = [w.lower() for w in tokens if w.isalnum() and w.lower() not in stop_words]
        freq  = {}
        for w in clean:
            freq[w] = freq.get(w, 0) + 1
        results["keywords"] = sorted(freq, key=freq.get, reverse=True)[:7]

    if "summarize" in actions:
        sentences = sent_tokenize(text)
        results["summary"] = " ".join(sentences[:2])
    return results

## Data Transformer Tool

In [ ]:
def data_transformer_tool(params: dict) -> dict:
    """
    Converts units, formats dates, validates emails.
    Params: { "type": "unit_convert|date_format|email_validate", ... }
    """
    ttype = params.get("type", "").lower()

    if ttype == "unit_convert":
        value = float(params.get("value", 0))
        from_unit = params.get("from_unit", "").lower()
        to_unit = params.get("to_unit", "").lower()

        table = {
            ("km", "miles"): lambda x: x * 0.621371,
            ("miles", "km"): lambda x: x * 1.60934,
            ("celsius", "fahrenheit"): lambda x: x * 9/5 + 32,
            ("fahrenheit", "celsius"): lambda x: (x - 32) * 5/9,
            ("kg", "lbs"): lambda x: x * 2.20462,
            ("lbs", "kg"): lambda x: x * 0.453592,
        }

        fn = table.get((from_unit, to_unit))
        
        if not fn:
            return {"error": f"Unsupported: {from_unit} -> {to_unit}"}

        return {
            "input": f"{value} {from_unit}",
            "result": f"{round(fn(value), 4)} {to_unit}"
        }

    elif ttype == "date_format":
        date_str = params.get("date", "")
        from_fmt = params.get("from_format", "%Y-%m-%d")
        to_fmt = params.get("to_format", "%B %d, %Y")

        try:
            dt = datetime.strptime(date_str, from_fmt)
            
            return {
                "original": date_str,
                "formatted": dt.strftime(to_fmt),
                "day": dt.strftime("%A")
            }
        except ValueError:
            return {"error": f"Cannot parse '{date_str}' with format '{from_fmt}'"}

    elif ttype == "email_validate":
        email = params.get("email", "")
        valid = bool(re.match(r'^[\w.+-]+@[\w-]+\.[\w.-]+$', email))
        return {
            "email": email,
            "valid": valid,
            "message": "Valid email." if valid else "Invalid email format."
        }

    else:
        return {"error": f"Unknown type '{ttype}'. Use: unit_convert, date_format, email_validate"}

TOOLS = {
    "calculator": calculator_tool,
    "text_processor": text_processor_tool,
    "data_transformer": data_transformer_tool,
}

## Main

In [ ]:
TOOLS = {
    "calculator"       : calculator_tool,
    "text_processor"   : text_processor_tool,
    "data_transformer" : data_transformer_tool,
}

## User Query
def classify_intent(user_input: str) -> dict:
    """LLM decides which tool to use and extracts parameters."""
    prompt = (
        "You are an AI agent. Given the user request, choose the right tool and extract parameters.\n\n"
        "Available tools:\n"
        "  calculator       → math expressions. Params: {\"expression\": \"10 * (3 + 2)\"}\n"
        "  text_processor   → text analysis.   Params: {\"text\": \"...\", \"actions\": [\"word_count\",\"reading_time\",\"keywords\",\"summarize\"]}\n"
        "  data_transformer → conversions.     Params: {\"type\": \"unit_convert|date_format|email_validate\", ...}\n\n"
        "Return ONLY a JSON object: {\"tool\": \"tool_name\", \"params\": {...}}\n"
        "No explanation. No markdown. JSON only.\n\n"
        f"User: {user_input}\n\nJSON:"
    )

   # Output + JSON
    raw = generator(prompt, max_new_tokens=200, do_sample=False)[0]["generated_text"]
    generated = raw[len(prompt):].strip()

    for pattern in [r'```json\s*(.*?)```', r'(\{.*\})', r'(\{.*)']:
        match = re.search(pattern, generated, re.DOTALL)

        if match:
            try:
                return json.loads(match.group(1).strip())
            
            except json.JSONDecodeError:
                continue
    return {"tool": None, "params": {}}


# Final respponse
def format_final_response(user_input: str, tool_name: str, result: dict) -> str:
    """LLM formats the tool result into a professional response."""
    prompt = (
        "Format a clear, professional 1-2 sentence response.\n"
        f"User asked: {user_input}\n"
        f"Tool used: {tool_name}\n"
        f"Result: {json.dumps(result)}\n\n"
        "Response:"
    )
    
    raw = generator(prompt, max_new_tokens=80, do_sample=False)[0]["generated_text"]
    return raw[len(prompt):].strip()


# Agent
def run_agent(user_input: str):
    print(f"\n{'='*60}")
    print(f"User   : {user_input}")
    
    decision  = classify_intent(user_input)
    tool_name = decision.get("tool")
    params    = decision.get("params", {})

    print(f"Decision → Tool: {tool_name} | Params: {params}")
    
    if tool_name not in TOOLS:
        print("Agent  : Sorry, I could not find a suitable tool for that request.")
        return

    tool_result = TOOLS[tool_name](params)
    print(f"Result  : {json.dumps(tool_result, indent=2)}")

    response = format_final_response(user_input, tool_name, tool_result)
    print(f"Agent  : {response}")

## Tests

In [ ]:
run_agent("What is 2000 divided by 5?")
run_agent("Calculate 2 to the power of 10")
run_agent("What is 1000 divided by 0?")

run_agent(
    "Analyze this: 'Artificial intelligence is transforming every industry. "
    "From healthcare to finance, AI is helping professionals work smarter.' "
    "Give me word count and keywords."
)

run_agent("Convert 10 km to miles")
run_agent("What is 30 Celsius in Fahrenheit?")
run_agent("Format the date 2026-04-06")
run_agent("Is kyrillos@university.com a valid email?")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



User   : What is 2000 divided by 5?


Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Decision → Tool: calculator | Params: {'expression': '2000 / 5'}
Result  : {
  "expression": "2000 / 5",
  "result": 400.0
}


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Agent  : To convert 30 degrees Celsius to Fahrenheit, you can use the formula (Celsius x 9/5) + 32, which results in 86.0°F. Response: The conversion of 30 degrees Celsius to Fahrenheit is 86.0°F using the formula (Celsius x 9/5) + 32.

User   : Format the date 2026-04-06


Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Decision → Tool: data_transformer | Params: {'type': 'date_format', 'format': '%Y-%m-%d'}
Result  : {
  "error": "Cannot parse '' with format '%Y-%m-%d'"
}


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Agent  : The user requested to format the date '2026-04-06', but the tool encountered an error because it cannot parse an empty string with the specified format. To provide a correct formatted date, please provide a valid date in the required format (YYYY-MM-DD). Response: The user requested to format the date '2026-04-06', but

User   : Is kyrillos@university.com a valid email?
Decision → Tool: email_validate | Params: {'email': 'kyrillos@university.com'}
Agent  : Sorry, I could not find a suitable tool for that request.
